Imports

In [1]:
import matplotlib.pyplot as plt
from onnx.tools.net_drawer import GetPydotGraph, GetOpNodeProducer
import numpy
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier, Dataset, train as train_lgbm
import onnxruntime as rt
from onnxconverter_common.data_types import FloatTensorType
from onnxmltools.convert import convert_lightgbm
import IPython.display as d

Load Data

In [2]:
import pandas as pd

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y)
clr = LGBMClassifier(verbosity=-1)
clr.fit(X_train, y_train)

d.display(pd.DataFrame(X, columns=[iris.feature_names]))
d.display(clr)

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2
...,...,...,...,...
145,6.7,3.0,5.2,2.3
146,6.3,2.5,5.0,1.9
147,6.5,3.0,5.2,2.0
148,6.2,3.4,5.4,2.3


LGBMClassifier(verbosity=-1)

Convert model to ONNX

In [6]:
initial_type = [("float_input", FloatTensorType([None, 4]))]
onx = convert_lightgbm(clr, initial_types=initial_type, decision_path=True, decision_leaf=True)

The maximum opset needed by this model is only 9.


Predict with native lgbm

In [ ]:
native_pred=clr.predict(X_test, pred_leaf=False)
display(native_pred)

array([[ 0,  0, -2, -1, -3,  0,  0,  1,  1, -1,  0,  2, -1, -1, -3],
       [ 0,  1,  5,  2, -3,  0,  0,  0,  0, -1,  0, -1, -1, -1, -3],
       [ 0,  0, -2, -1, -3,  0,  0,  2,  1, -1,  0,  0, -1, -1, -3],
       [ 0,  0,  5,  2, -3, -1,  0,  0,  0, -1,  0,  0, -1, -1, -3],
       [ 0,  0,  5,  2, -3,  0,  0,  0,  0, -1,  0,  0, -1, -1, -3],
       [ 0,  0, -2, -1, -3,  0,  0, -1,  0, -1,  0,  0,  1,  2, -3],
       [ 0,  0,  5,  2, -3,  0,  0,  0,  0, -1,  0, -1, -1, -1, -3],
       [ 0,  0, -2, -1, -3,  0,  0,  1,  1, -1,  0,  2, -1, -1, -3],
       [ 0,  0, -2, -1, -3,  1,  0, -2,  0, -1,  0,  0,  3,  2, -3],
       [ 0,  2, -2, -1, -3,  1,  0, -2,  0, -1,  0,  0,  3,  2, -3],
       [ 0,  0,  5,  2, -3,  0,  0,  0,  0, -1,  0,  0, -1, -1, -3],
       [ 0,  0, -2, -1, -3,  1,  0, -1,  0, -1,  0,  0,  3,  2, -3],
       [ 0,  1,  5,  2, -3,  0,  0,  0,  0, -1,  0, -1, -1, -1, -3],
       [ 0,  0, -2, -1, -3,  0,  0,  2,  1, -1,  0,  2, -1, -1, -3],
       [ 0,  0, -2, -1, -3,  0,  0

Predict in onx session

In [8]:
sess = rt.InferenceSession(onx.SerializeToString(), providers=["CPUExecutionProvider"])
input_name = sess.get_inputs()[0].name
pred_onx = sess.run(['label', 'decision_path', 'decision_leaf'], {input_name: X_test.astype(numpy.float32)})
d.display(pd.DataFrame(pred_onx[2]))

2025-02-11 01:40:40.375370 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {38} for output label


,0
0,1601
1,1383
2,1961
3,1371
4,1343
5,1770
6,1403
7,1634
8,1854
9,1624
